# LangChain PDF Loading & RAG

#### Week 2 deliverable - 3nd Iteration.
####  This notebook also reads/chunks the PDF documents.  However, we are using LangGraph to perform the PDF reading/chunking. 

#### Interesting new concepts for this iteration:
 - Introduced LangChain to read and chunk the PDFs
 - Add an interactive query (compare) method so that the user can interactively query the RAG database.


Replaces the custom `pypdf` + regex section splitter with LangChain's document pipeline:

| Step | LangChain component |
|---|---|
| Load PDFs | `PyPDFLoader` — loads each page as a `Document` with metadata |
| Split into chunks | `RecursiveCharacterTextSplitter` — splits on paragraphs → sentences → words with configurable overlap |

The same three ChromaDB collections and `compare()` function from the previous notebook are used to query the results.


In [1]:
import os
import tiktoken
import huggingface_hub
import chromadb
import openai as openai_client
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import (
    DefaultEmbeddingFunction,
    SentenceTransformerEmbeddingFunction,
    OpenAIEmbeddingFunction,
)

load_dotenv(find_dotenv())

try:
    huggingface_hub.login(token=os.environ.get("HUGGINGFACE_API_KEY"), add_to_git_credential=False)
    print("HuggingFace: authenticated")
except Exception as e:
    print(f"HuggingFace: proceeding without auth ({e})")

/var/folders/1f/srb1v76j181_9p65kq4xmmjw0000gn/T/ipykernel_6236/2954100493.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


HuggingFace: proceeding without auth (Invalid user token.)


In [2]:
# INGEST:  This Python cell loads the PDF files from the data directory and extracts the sections.
data_dir = Path("data")

# PyPDFLoader loads each page as a separate LangChain Document with metadata
# (source filename, page number).  We load all PDFs and collect their pages.
raw_docs = []
for pdf_path in sorted(data_dir.glob("*.pdf")):
    loader = PyPDFLoader(str(pdf_path))
    raw_docs.extend(loader.load())

print(f"Loaded {len(raw_docs)} pages from {len(list(data_dir.glob('*.pdf')))} PDFs")

# RecursiveCharacterTextSplitter tries to split on paragraphs first (\n\n),
# then sentences (\n), then words — preserving as much semantic coherence as
# possible while keeping chunks within chunk_size characters.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],
)

all_chunks = splitter.split_documents(raw_docs)

# Normalise into the same list-of-dicts format the rest of the notebook uses
all_sections = [
    {
        "text": doc.page_content,
        "source": Path(doc.metadata.get("source", "")).name,
        "chunk_index": i,
        "heading": doc.page_content.splitlines()[0].strip()[:80],
    }
    for i, doc in enumerate(all_chunks)
]

print(f"Split into {len(all_sections)} chunks (avg {sum(len(s['text']) for s in all_sections) // len(all_sections)} chars/chunk)")

Loaded 546 pages from 3 PDFs
Split into 1844 chunks (avg 840 chars/chunk)


In [3]:
# INDEXING:  This Python cell embeds the RAG text using three different embedding models. 
#### We are building three collections to evaluate all three embedding models. 
import openai as openai_client

OPENAI_MAX_TOKENS = 7_500  # safely under the 8191 token limit for text-embedding-3-small
OPENAI_BATCH_SIZE = 500    # stay well under OpenAI's 2048 inputs-per-request limit
_enc = tiktoken.get_encoding("cl100k_base")

# This function sanitizes the text by removing any null bytes and encoding it to UTF-8.
# It also limits the number of tokens to the maximum number of tokens allowed by the model.
def sanitize(text: str, max_tokens: int | None = None) -> str:
    text = text.replace("\x00", "").encode("utf-8", errors="ignore").decode("utf-8").strip()
    if max_tokens:
        tokens = _enc.encode(text)
        if len(tokens) > max_tokens:
            text = _enc.decode(tokens[:max_tokens])
    return text

# This function embeds the text using the OpenAI API.
# It embeds the text in batches of 500 to stay under the 2048 inputs-per-request limit.
def embed_openai_batched(docs: list[str], model: str = "text-embedding-3-small") -> list[list[float]]:
    client_oa = openai_client.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    embeddings = []
    for i in range(0, len(docs), OPENAI_BATCH_SIZE):
        batch = docs[i:i + OPENAI_BATCH_SIZE]
        response = client_oa.embeddings.create(input=batch, model=model)
        embeddings.extend([r.embedding for r in response.data])
        print(f"  OpenAI: embedded {min(i + OPENAI_BATCH_SIZE, len(docs))}/{len(docs)} docs")
    return embeddings

chroma_client = chromadb.EphemeralClient()
collections = {}

# This loop builds two collections to evaluate the default ChromaDB embedding model and the HuggingFace SentenceTransformer embedding model. 
# --- Store embeddings for local models:  all-MiniLM-L6-v2 and all-mpnet-base-v2 ---
for model_name, ef in [
    ("minilm (default)", DefaultEmbeddingFunction()),
    ("mpnet (huggingface)", SentenceTransformerEmbeddingFunction(model_name="all-mpnet-base-v2")),
]:
    docs = [sanitize(s["text"]) for s in all_sections]
    col = chroma_client.get_or_create_collection(
        name=model_name.replace(" ", "_").replace("(", "").replace(")", ""),
        embedding_function=ef,
    )
    col.add(
        documents=docs,
        metadatas=[{"source": s["source"], "chunk_index": s["chunk_index"], "heading": s["heading"]} for s in all_sections],
        ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in all_sections],
    )
    collections[model_name] = col
    print(f"Built collection: {model_name}")

# --- OpenAI model (pre-computed embeddings in controlled batches) ---
openai_docs = [sanitize(s["text"], OPENAI_MAX_TOKENS) for s in all_sections]
print("Building OpenAI embeddings...")
openai_embeddings = embed_openai_batched(openai_docs)
col = chroma_client.get_or_create_collection(
    name="text_embedding_3_small_openai",
    embedding_function=OpenAIEmbeddingFunction(api_key=os.environ["OPENAI_API_KEY"], model_name="text-embedding-3-small"),
)
col.add(
    documents=openai_docs,
    embeddings=openai_embeddings,
    metadatas=[{"source": s["source"], "chunk_index": s["chunk_index"], "heading": s["heading"]} for s in all_sections],
    ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in all_sections],
)
collections["text-embedding-3-small (openai)"] = col
print("Built collection: text-embedding-3-small (openai)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Built collection: minilm (default)
Built collection: mpnet (huggingface)
Building OpenAI embeddings...
  OpenAI: embedded 500/1844 docs
  OpenAI: embedded 1000/1844 docs
  OpenAI: embedded 1500/1844 docs
  OpenAI: embedded 1844/1844 docs
Built collection: text-embedding-3-small (openai)


In [4]:
# RETRIEVAL:  This Python cell embeds the queries and returns the SNIPPET_CHARS results. 
# We will use this function to compare the results of the three models. (in the collections array)
SNIPPET_CHARS = 1000

# This function compares the results of the three models.
# It takes a query and the number of results to return.
# It then prints the results side-by-side for the three models.
def compare(query: str, n_results: int = 5):
    print(f"\nQUERY: {query}\n{'=' * 80}")
    for model_name, col in collections.items():
        results = col.query(query_texts=[query], n_results=n_results)
        print(f"\n--- {model_name.upper()} ---")
        for text, meta, distance in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        ):
            snippet = " ".join(text.split())[:SNIPPET_CHARS]
            print(f"  Score: {1 - distance:.3f} | {meta['source']} — {meta['heading']}")
            print(f"    {snippet}...")

compare("do i have to register with the boro if i want to install an alarm?")
compare("what are the rules for parking on residential streets?")
compare("how do i appeal a zoning decision?")
compare("What company is authorized to provide and maintain the telephone service within the boro?")
compare("I'd like to put an alarm system in my business in the borough.  What are the requirements to do this?")
compare("Can I put solar panels on my house roof?  What are the steps for approval?") 
compare("I'm a police officer and would like to work for the borough until I'm 67 years old.  Is this possible?")
compare("I have a construction company, how can I be included on the list of qualified suppliers for borough contracts? ")
compare("What is the overtime policy for the borough?")
compare("What paid holidays does the borough observe?")
compare("what is the leaf removal policy for the borough?")
compare("Where can I find information about the borough's recycling program?")
compare("Where can I find information on the borough's building codes for residential construction?")
compare("What are the requirements for a business license in the borough?")
compare("what does the code enforcement officier do")
compare("What does the building inspector do?")
compare("Can I have a fire pit in my backyard?")        







QUERY: do i have to register with the boro if i want to install an alarm?

--- MINILM (DEFAULT) ---
  Score: 0.189 | BaldwinBoro-GeneralLegislation.pdf — §  62-4. Alarm systems in apartment complexes.
    § 62-4. Alarm systems in apartment complexes. A tenant of an apartment with an alarm system shall obtain an alarm registration from Baldwin Borough before operating or causing the operation of an alarm system in the tenant's residential unit. The owner or property manager of an apartment complex shall obtain a separate alarm registration of any alarm system operated in offices or common areas of the apartment complex. The annual fee for these registrations or the renewal of these registrations shall be the same as the fee for a commercial alarm site. Each building, regardless of the number of units contained therein, shall be deemed as one commercial alarm site and shall pay the annual registration fee for a commercial alarm site. § 62-5. Registration duration and renewal. A registra

In [5]:
import anthropic

# Conversation memory — grows with each turn and is sent as context on every request.
# Inspect it at any time, or reset with: conversation_history.clear()
conversation_history: list[dict] = []

def compare_interactive(
    collection_name: str = "text-embedding-3-small (openai)",
    n_results: int = 5,
):
    """
    Interactive RAG chat session with conversation memory.

    Retrieves the top n_results chunks from the chosen collection,
    builds a prompt that includes those chunks plus the full conversation
    history, and sends it to Claude to generate a grounded answer.

    Commands during the session:
      quit / exit / q  — end the session
      clear            — wipe conversation memory and start fresh
    """
    global conversation_history
    col = collections[collection_name]
    ac = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    print(f"RAG Chat  |  collection: '{collection_name}'")
    print("Commands: 'quit' to exit  |  'clear' to reset memory\n")

    while True:
        query = input("You: ").strip()

        if not query:
            continue
        if query.lower() in ("quit", "exit", "q"):
            print("Session ended.")
            break
        if query.lower() == "clear":
            conversation_history.clear()
            print("[Memory cleared]\n")
            continue

        # --- retrieve relevant chunks ---
        results = col.query(query_texts=[query], n_results=n_results)
        context_str = "\n\n".join(
            f"[{meta['source']} — {meta['heading']}]\n{text}"
            for text, meta in zip(results["documents"][0], results["metadatas"][0])
        )

        # --- build message list from growing history ---
        messages = []
        for turn in conversation_history:
            messages.append({"role": "user",      "content": turn["question"]})
            messages.append({"role": "assistant",  "content": turn["answer"]})
        messages.append({"role": "user", "content": query})

        # --- call Claude with RAG context as the system prompt ---
        response = ac.messages.create(
            model="claude-opus-4-5",
            max_tokens=1024,
            system=(
                "You are a helpful assistant answering questions about Baldwin Borough "
                "municipal regulations. Answer using only the context below. "
                "If the answer is not in the context, say so clearly.\n\n"
                f"CONTEXT:\n{context_str}"
            ),
            messages=messages,
        )

        answer = response.content[0].text
        print(f"\nAssistant: {answer}")

        # --- citation footer ---
        seen = set()
        citations = []
        for meta in results["metadatas"][0]:
            key = (meta["source"], meta["heading"])
            if key not in seen:
                seen.add(key)
                citations.append(f"  • {meta['source']} — {meta['heading']}")
        print("\n" + "─" * 60)
        print("Sources:")
        print("\n".join(citations))
        print("─" * 60)

        print(f"\n[Memory: {len(conversation_history) + 1} turn(s)]\n")

        conversation_history.append({"question": query, "answer": answer})


# Run the session — re-execute this cell to start a new session (memory is preserved
# across re-runs unless you call conversation_history.clear() first).
compare_interactive()

RAG Chat  |  collection: 'text-embedding-3-small (openai)'
Commands: 'quit' to exit  |  'clear' to reset memory


Assistant: # Fire Pits in Baldwin Borough

**Yes, you can use a fire pit in your backyard**, but you must follow these rules:

## Requirements for Outdoor Fire Pits:

1. **Approved container required** - Burning must be in an approved outdoor-rated fireplace, fire pit, chimney, or 55-gallon drum

2. **Fuel restrictions** - Only clean, dry wood products may be burned. No treated wood or wood products that produce noxious or unpleasant odors

3. **Distance from property line** - The fire must be at least **15 feet** from the nearest property line

4. **Adult supervision** - The fire must be attended at all times by someone **18 years or older** and must be extinguished if no qualified adult is present

5. **Fire extinguishment** - You must have adequate means to extinguish the fire immediately available

6. **Safe conditions** - Burning should only occur when wind and weather